[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusicalInformatics/miws26/blob/main/alignment_1/intro_to_spectrograms_and_chroma.ipynb)

# Spectrograms and Chroma Features

In the previous notebook (the introduction to Fourier analysis), we computed the Fourier transform of a whole signal. This tells us *which* frequencies are in a signal, but not *when* they occur. In music, where notes change over time, we need both. In this notebook we will:

1. see why the Fourier transform of a whole signal is not enough,
2. introduce the **short-time Fourier transform (STFT)** and the **spectrogram**,
3. look at the effect of the **window** (its type and length) on the spectrogram,
4. convert a spectrogram into a **log-frequency spectrogram**, whose frequency axis is labeled with pitches, and
5. fold it into a **chromagram**, a feature that captures the harmonic content of music, and that is the basis of many music alignment methods.

> **Note.** This notebook is based on the following notebooks from the *Fundamentals of Music Processing* (FMP) notebooks by Meinard Müller and Frank Zalkow:
>
> - [Discrete Short-Time Fourier Transform (STFT)](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C2/C2_STFT-Basic.html)
> - [STFT: Influence of Window Function](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C2/C2_STFT-Window.html)
> - [STFT: Conventions and Implementations](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C2/C2_STFT-Conventions.html)
> - [Log-Frequency Spectrogram and Chromagram](https://www.audiolabs-erlangen.de/resources/MIR/FMP/C3/C3S1_SpecLogFreq-Chromagram.html)
>
> We follow the structure and the main ideas of these notebooks, but the text, the examples and the code have been rewritten for the purposes of this class (using the packages in our environment instead of `libfmp`). For more details, see the references at the end of this notebook.

In [ ]:
import os

try:
    import google.colab

    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    # Install python packages
    ! pip install git+https://github.com/CPJKU/partitura.git@develop

    # To be able to access helper modules and data in the repo for this tutorial
    # (not necessary if the jupyter notebook is run locally instead of google colab)
    if not os.path.exists("./miws26"):
        !git clone https://github.com/MusicalInformatics/miws26.git
    import sys

    sys.path.insert(0, "./miws26/alignment_1/")

# Path to the mec-variation dataset, which contains the music example that we use
if IN_COLAB:
    MEC_VARIATION_DIR = "./miws26/alignment_1/mec-variation"
else:
    # The notebook is in the alignment_1 folder, next to the dataset
    MEC_VARIATION_DIR = os.path.join(os.path.abspath("."), "mec-variation")

In [ ]:
import warnings
from typing import Callable, Dict, Tuple

import IPython.display as ipd
import ipywidgets as widgets
import librosa
import matplotlib.pyplot as plt
import numpy as np
import partitura as pt
from matplotlib.axes import Axes
from matplotlib.figure import Figure

from helper import interactive_figure

warnings.filterwarnings("ignore")

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. The Fourier transform has no time information

Consider two signals that are made of the same two notes, A4 (440 Hz) and E5 (660 Hz), each lasting half a second. In the first signal, A4 comes first, and in the second one, E5 comes first. Listen to them! They sound very different. Now look at their magnitude spectra:

In [ ]:
DEMO_SAMPLE_RATE = 8000  # sample rate of the demo signals (in Hz)


def sine(freq: float, duration: float, sample_rate: float = DEMO_SAMPLE_RATE) -> np.ndarray:
    """A sine wave with a given frequency (in Hz) and duration (in seconds)"""
    t = np.arange(int(duration * sample_rate)) / sample_rate
    return np.sin(2 * np.pi * freq * t)


x_up = np.concatenate([sine(440, 0.5), sine(660, 0.5)])  # A4, then E5
x_down = np.concatenate([sine(660, 0.5), sine(440, 0.5)])  # E5, then A4

print("A4, then E5")
ipd.display(ipd.Audio(x_up, rate=DEMO_SAMPLE_RATE))
print("E5, then A4")
ipd.display(ipd.Audio(x_down, rate=DEMO_SAMPLE_RATE))

fig, axes = plt.subplots(2, 2, figsize=(10, 4.5))
for j, (x, title) in enumerate([(x_up, "A4, then E5"), (x_down, "E5, then A4")]):
    t = np.arange(len(x)) / DEMO_SAMPLE_RATE
    axes[0, j].plot(t, x, color="navy", linewidth=0.8)
    axes[0, j].set_xlim(0.47, 0.53)  # zoom around the change of note
    axes[0, j].set_title(title + " (waveform, zoomed around 0.5 s)")
    axes[0, j].set_xlabel("Time (s)")
    axes[0, j].set_ylabel("Amplitude")
    freqs = np.fft.rfftfreq(len(x), 1 / DEMO_SAMPLE_RATE)
    axes[1, j].plot(freqs, np.abs(np.fft.rfft(x)), color="firebrick")
    axes[1, j].set_xlim(0, 1000)
    axes[1, j].set_xlabel("Frequency (Hz)")
    axes[1, j].set_ylabel("Magnitude")
plt.tight_layout()
plt.show()

The two magnitude spectra are (almost) the same: both show a peak at 440 Hz and a peak at 660 Hz. The information about *when* each note is played is hidden in the *phase* of the Fourier transform, where it is very hard to read. To make it visible, we need to look at the signal locally, in short sections.

## 2. The short-time Fourier transform

### 2.1 Basic idea

The main idea of the **short-time Fourier transform (STFT)**, which was introduced by Dennis Gabor in 1946, is to look at only a small section of the signal at a time:

1. Choose a **window function**: a function that is nonzero only for a short period of time.
2. Multiply the signal with the window, to cut out a section of the signal.
3. Compute the Fourier transform of this section.
4. Shift the window along the signal and repeat.

Try it out! In the plot below, the gray line is the signal with A4 followed by E5. The blue line is the part of the signal that we see through the window, and the plot at the bottom is its spectrum. Move the window along the signal, and change its length and its shape (we use some of the standard window functions that are implemented in numpy). What do you observe when the window covers only one of the notes? And when it covers both?

In [ ]:
# Some of the standard window functions that are implemented in numpy
WINDOW_FUNCTIONS: Dict[str, Callable[[int], np.ndarray]] = {
    "rectangular": np.ones,
    "bartlett": np.bartlett,  # triangular window
    "hann": np.hanning,
    "hamming": np.hamming,
    "blackman": np.blackman,
    "kaiser": lambda length: np.kaiser(length, 8),  # the parameter beta (here, 8) controls the shape
}


def make_window(window: str, length: int) -> np.ndarray:
    """
    Window of a given type and length (in samples).

    The windows in numpy are symmetric. For spectral analysis, we use the periodic
    version of the window (like librosa and scipy do), which is the symmetric window
    with one sample more, without its last sample.
    """
    return WINDOW_FUNCTIONS[window](length + 1)[:-1]


def windowed_spectrum_figure(center: float, length_ms: float, window: str) -> Figure:
    """Figure with the signal (and the window) and the spectrum of the windowed signal"""
    x, sample_rate = x_up, DEMO_SAMPLE_RATE
    t = np.arange(len(x)) / sample_rate

    # Build the window (centered at `center` seconds)
    win_len = max(int(length_ms / 1000 * sample_rate), 2)
    start = int(round(center * sample_rate)) - win_len // 2
    w = np.zeros(len(x))
    w_short = make_window(window, win_len)
    lo, hi = max(start, 0), min(start + win_len, len(x))
    w[lo:hi] = w_short[lo - start : hi - start]
    x_windowed = x * w

    # Spectrum of the windowed signal (zero-padded for a smooth plot)
    n_fft = 2**15
    freqs = np.fft.rfftfreq(n_fft, 1 / sample_rate)
    spectrum = np.abs(np.fft.rfft(x_windowed, n_fft))

    fig = Figure(figsize=(9, 4.5))
    ax1, ax2 = fig.subplots(2, 1)
    ax1.plot(t, x, color="lightgray", label="signal")
    ax1.plot(t, x_windowed, color="navy", label="windowed signal")
    ax1.plot(t, w, color="darkorange", linestyle="--", label="window")
    ax1.set_xlim(0, 1)
    ax1.set_ylim(-1.2, 1.2)
    ax1.set_xlabel("Time (s)")
    ax1.legend(loc="upper right", ncol=3, fontsize=8)
    ax2.plot(freqs, spectrum, color="firebrick")
    ax2.set_xlim(0, 1000)
    ax2.set_ylim(0, 2100)  # the highest peak: a note that fills the whole window (0.5 s)
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Magnitude")
    fig.tight_layout()
    return fig


window_controls = {
    "center": widgets.FloatSlider(
        value=0.25, min=0, max=1, step=0.01, description="position (s)", style={"description_width": "100px"}
    ),
    "length_ms": widgets.FloatSlider(
        value=250, min=20, max=1000, step=10, description="length (ms)", style={"description_width": "100px"}
    ),
    "window": widgets.Dropdown(
        options=list(WINDOW_FUNCTIONS), value="rectangular", description="window", style={"description_width": "100px"}
    ),
}
ipd.display(widgets.VBox(list(window_controls.values())), interactive_figure(windowed_spectrum_figure, window_controls))

When the window covers only one note, the spectrum shows only one peak, and we know which note is played in that section. When the window covers both notes, we get both peaks again. So with a window that is short enough, we can tell *what* is played *when*. The result depends on the window: its **length** determines the size of the section, and its **shape** has a strong influence on the spectrum (we will look into this in Section 3).

### 2.2 Definition

Let $x$ be a signal of length $L$, sampled with a sampling rate $F_\mathrm{s}$, let $w$ be a window of length $N$, and let $H$ be the **hop size**, i.e., the number of samples that the window is shifted at each step. The (discrete) STFT is given by

$$\mathcal{X}(m, k) = \sum_{n=0}^{N-1} w(n)\, x\left(n + mH - \tfrac{N}{2}\right) e^{-2\pi i k n / N},$$

where $m$ is the **frame** (the position of the window), $k \in [0:N/2]$ is the frequency index, and $x(n) = 0$ if $n$ is outside of the signal (the signal is zero-padded). For real signals, the upper half of the frequency coefficients are redundant, so we only keep $k \in [0:N/2]$. The squared magnitude of the STFT is called the **spectrogram**,

$$\mathcal{Y}(m, k) = |\mathcal{X}(m, k)|^2.$$

### 2.3 Time and frequency axes

In the definition above, the window of frame $m$ is *centered* at sample $mH$ (this is why we shift the signal by $N/2$). Therefore, the coefficient $\mathcal{X}(m, k)$ is associated with the time (in seconds) and frequency (in Hz)

$$T(m) = \frac{m H}{F_\mathrm{s}}, \qquad F(k) = \frac{k F_\mathrm{s}}{N}.$$

The index $k = N/2$ corresponds to the Nyquist frequency $F_\mathrm{s}/2$. Note that the time resolution is $H / F_\mathrm{s}$, and the frequency resolution is $F_\mathrm{s} / N$.

### 2.4 Implementation

Let's implement the STFT!

In [ ]:
def stft(x: np.ndarray, window_length: int, hop_size: int, window: str = "hann") -> np.ndarray:
    """
    Short-time Fourier transform of a real signal (centered, with zero padding).

    Parameters
    ----------
    x : np.ndarray
        Signal (1D array).
    window_length : int
        Length N of the window (in samples). It should be even.
    hop_size : int
        Hop size H (in samples).
    window : str
        Type of the window (one of the keys of `WINDOW_FUNCTIONS`).

    Returns
    -------
    X : np.ndarray
        Complex array with shape (N / 2 + 1, number of frames): frequency index k
        in the rows and frame index m in the columns.
    """
    w = make_window(window, window_length)
    # Zero padding of half a window on each side: frame m is centered at sample m * H
    x_padded = np.pad(x, window_length // 2)
    n_frames = 1 + len(x) // hop_size
    X = np.zeros((window_length // 2 + 1, n_frames), dtype=complex)
    for m in range(n_frames):
        frame = x_padded[m * hop_size : m * hop_size + window_length]
        X[:, m] = np.fft.rfft(w * frame)
    return X


def frame_times(n_frames: int, hop_size: int, sample_rate: float) -> np.ndarray:
    """Time (in seconds) of each frame of the STFT"""
    return np.arange(n_frames) * hop_size / sample_rate


def bin_frequencies(window_length: int, sample_rate: float) -> np.ndarray:
    """Frequency (in Hz) of each frequency bin of the STFT"""
    return np.arange(window_length // 2 + 1) * sample_rate / window_length


def power_to_db(Y: np.ndarray, dynamic_range: float = 80) -> np.ndarray:
    """Convert a power spectrogram to decibels (limited to a dynamic range)"""
    Y_db = 10 * np.log10(np.maximum(Y, 1e-12))
    return np.maximum(Y_db, Y_db.max() - dynamic_range)


def plot_spectrogram(
    ax: Axes,
    Y: np.ndarray,
    sample_rate: float,
    window_length: int,
    hop_size: int,
    fmax: float = None,
    fmin: float = 0,
    cmap: str = "magma",
) -> None:
    """Plot a spectrogram (an image with time on the x axis and frequency on the y axis)"""
    T = frame_times(Y.shape[1], hop_size, sample_rate)
    F = bin_frequencies(window_length, sample_rate)
    # The extent centers each pixel at its time and frequency
    extent = [
        T[0] - hop_size / sample_rate / 2,
        T[-1] + hop_size / sample_rate / 2,
        F[0] - sample_rate / window_length / 2,
        F[-1] + sample_rate / window_length / 2,
    ]
    ax.imshow(Y, origin="lower", aspect="auto", extent=extent, cmap=cmap)
    ax.set_ylim(fmin, fmax if fmax is not None else sample_rate / 2)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")


# Compare with librosa
N, H = 256, 64
X = stft(x_up, N, H)
X_librosa = librosa.stft(x_up, n_fft=N, hop_length=H, win_length=N, window="hann", center=True, pad_mode="constant")
print("Same result as librosa.stft:", np.allclose(X, X_librosa, atol=1e-3))
print("Same time axis as librosa:", np.allclose(frame_times(X.shape[1], H, DEMO_SAMPLE_RATE), librosa.frames_to_time(np.arange(X.shape[1]), sr=DEMO_SAMPLE_RATE, hop_length=H)))
print("Same frequency axis as librosa:", np.allclose(bin_frequencies(N, DEMO_SAMPLE_RATE), librosa.fft_frequencies(sr=DEMO_SAMPLE_RATE, n_fft=N)))
print(f"Shape of the STFT: {X.shape[0]} frequency bins x {X.shape[1]} frames")

Our implementation gives the same result as `librosa.stft` (with the settings `center=True` and `pad_mode="constant"`), and the same time and frequency axes. Now let's compute the spectrogram of the signal with A4 followed by E5. (Remember that the window is a Hann window, with length 256 samples, i.e., 32 ms, and the hop size is 64 samples, i.e., 8 ms.)

In [ ]:
Y = np.abs(X) ** 2

fig, ax = plt.subplots(figsize=(7, 3))
plot_spectrogram(ax, Y, DEMO_SAMPLE_RATE, N, H, fmax=1000)
ax.set_title("Spectrogram: A4, then E5")
plt.tight_layout()
plt.show()

Now we see in the image *what* is played *when*: a horizontal line at 440 Hz during the first half second, followed by a horizontal line at 660 Hz.

### 2.5 A music example

Let's look at a real recording: the first two measures of the first variation of Mozart's *"Ah! vous dirai-je, maman"* (K. 265) played on a piano, from the `mec-variation` dataset (we also use this example in the other notebooks in this folder).

In [ ]:
SAMPLE_RATE = 22050  # we use a lower sample rate than the original recording

x_piano, _ = librosa.load(os.path.join(MEC_VARIATION_DIR, "performances", "wav", "snippet.wav"), sr=SAMPLE_RATE)
ipd.display(ipd.Audio(x_piano, rate=SAMPLE_RATE))

fig, ax = plt.subplots(figsize=(7, 2))
ax.plot(np.arange(len(x_piano)) / SAMPLE_RATE, x_piano, color="navy", linewidth=0.5)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
plt.tight_layout()
plt.show()

The waveform shows when notes start (the loud parts), but it is hard to tell which notes are played. Let's compute its spectrogram, with a window of 2048 samples (about 93 ms) and a hop size of 512 samples (about 23 ms). The values of a spectrogram have a very large range: the energy in the low frequencies is much larger than in the high frequencies. Because of this (and because our perception of loudness is roughly logarithmic), we usually plot the spectrogram on a logarithmic scale, in **decibels** (dB), $10 \log_{10}(\mathcal{Y})$. Compare the two plots:

In [ ]:
N, H = 2048, 512
X_piano = stft(x_piano, N, H)
Y_piano = np.abs(X_piano) ** 2

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
plot_spectrogram(axes[0], Y_piano, SAMPLE_RATE, N, H, fmax=4000)
axes[0].set_title("Spectrogram (linear scale)")
plot_spectrogram(axes[1], power_to_db(Y_piano), SAMPLE_RATE, N, H, fmax=4000)
axes[1].set_title("Spectrogram (in dB)")
plt.tight_layout()
plt.show()

On the linear scale, we only see the strongest components. On the dB scale, we also see the weaker ones: each note appears as a stack of horizontal lines. The lowest line is the fundamental frequency, and the lines above it are its harmonics (the integer multiples of the fundamental, as we saw in the previous notebook).

## 3. The influence of the window

The STFT does not only depend on the signal, but also on the window. We look at two properties of the window: its type and its length.

### 3.1 Window type

The simplest window is the **rectangular window**: it leaves the signal unchanged inside the section, and sets it to zero outside. However, it cuts the signal abruptly at the boundaries of the section, and these discontinuities introduce artifacts in the spectrum ("leakage": the energy of a frequency spreads to other frequencies). To avoid this, we can use windows that go smoothly to zero at the boundaries. Numpy implements several of the standard ones: the triangular window (`np.bartlett`), and the `np.hanning`, `np.hamming`, `np.blackman` and `np.kaiser` windows. The plot below shows these windows (left), and the spectra (in dB) of a sinusoid at 440.5 Hz seen through each of them (right):

In [ ]:
sample_rate = 8000
n_win = 512
t = np.arange(n_win) / sample_rate
x_sine = np.sin(2 * np.pi * 440.5 * t)  # the section does not contain a whole number of periods

n_fft = 2**15
freqs = np.fft.rfftfreq(n_fft, 1 / sample_rate)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for window in WINDOW_FUNCTIONS:
    w = make_window(window, n_win)
    axes[0].plot(w, label=window)
    spectrum = np.abs(np.fft.rfft(w * x_sine, n_fft))
    axes[1].plot(freqs, 20 * np.log10(spectrum / spectrum.max() + 1e-9), label=window)
axes[0].set_title("Windows")
axes[0].set_xlabel("Sample")
axes[0].legend()
axes[1].set_title("Spectrum of the windowed sinusoid (in dB)")
axes[1].set_xlim(300, 600)
axes[1].set_ylim(-100, 5)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Magnitude (dB)")
plt.tight_layout()
plt.show()

Two properties of the spectrum of a window determine how it behaves: the width of its **main lobe** (which determines how close two frequencies can be to be separated), and the height of its **sidelobes** (the ripples around the main lobe, which determine how much a strong frequency leaks into weaker ones nearby). We can measure them from the spectrum of each window:

In [ ]:
def window_lobes(window: str, length: int = 512, padding: int = 64) -> Tuple[float, float]:
    """Width of the main lobe (in frequency bins) and highest sidelobe (in dB) of a window"""
    W = np.abs(np.fft.rfft(make_window(window, length), length * padding))
    W_db = 20 * np.log10(W / W.max() + 1e-12)
    first_null = 1
    while not (W_db[first_null] <= W_db[first_null - 1] and W_db[first_null] <= W_db[first_null + 1]):
        first_null += 1
    return 2 * first_null / padding, W_db[first_null:].max()


print(f"{'window':12s} {'main lobe width (bins)':>24s} {'highest sidelobe (dB)':>24s}")
for window in WINDOW_FUNCTIONS:
    width, sidelobe = window_lobes(window)
    print(f"{window:12s} {width:24.1f} {sidelobe:24.1f}")

The rectangular window has the narrowest main lobe (2 bins), but the highest sidelobes (only 13 dB below the peak). The other windows have a wider main lobe, but much lower sidelobes: the triangular (Bartlett) and the Hann windows have a main lobe that is twice as wide, and sidelobes that are 26 dB and 31 dB below the peak. The Hamming window is a small variation of the Hann window, with a lower first sidelobe (43 dB below the peak), but whose sidelobes decay more slowly far away from the peak (look at the right of the plot above). The Blackman window has an even lower first sidelobe (58 dB below the peak), at the price of a main lobe that is three times as wide as the one of the rectangular window. The Kaiser window has a parameter $\beta$ which controls this trade-off: with $\beta = 0$ it is the rectangular window, and the larger $\beta$ is, the lower the sidelobes and the wider the main lobe (here, $\beta = 8$). There is no best window: it is a trade-off between frequency resolution and leakage. For spectrograms of music, the Hann window is a standard choice.

You can compare the effect of these windows on the spectrogram with the interactive plot below.

### 3.2 Window length

The length of the window has a very strong effect: it determines the balance between **time resolution** and **frequency resolution**. With a window of length $N$ (in samples),

- the frequency resolution is about $F_\mathrm{s} / N$ Hz: a *longer* window can separate frequencies that are *closer*, but
- a longer window spans more time, so events that are close in time are blurred together.

To see this, we use a test signal with two simultaneous sinusoids that are one semitone apart (A4 at 440 Hz and A♯4 at about 466 Hz), and two short clicks that are 30 ms apart. We also have the piano example from above. Change the length of the window! Which window lengths separate the two sinusoids? Which ones separate the two clicks? Can you find a window length that separates both?

In [ ]:
TEST_SAMPLE_RATE = 8000
t = np.arange(TEST_SAMPLE_RATE) / TEST_SAMPLE_RATE
x_test = np.sin(2 * np.pi * 440 * t) + np.sin(2 * np.pi * 466.16 * t)  # A4 and A#4
x_test[int(0.40 * TEST_SAMPLE_RATE)] = 20  # two clicks, 30 ms apart
x_test[int(0.43 * TEST_SAMPLE_RATE)] = 20

# Name -> (signal, sample rate, range of frequencies to show)
SIGNALS: Dict[str, Tuple[np.ndarray, float, Tuple[float, float]]] = {
    "Test signal (two sinusoids and two clicks)": (x_test, TEST_SAMPLE_RATE, (350, 600)),
    "Piano example": (x_piano, SAMPLE_RATE, (0, 3000)),
}

print("Test signal")
ipd.display(ipd.Audio(x_test, rate=TEST_SAMPLE_RATE))


def window_length_figure(signal_name: str, window_ms: float, window: str) -> Figure:
    """Figure with a signal and its spectrogram, computed with a given window"""
    x, sample_rate, (fmin, fmax) = SIGNALS[signal_name]
    n_win = 2 * max(int(round(window_ms / 1000 * sample_rate / 2)), 4)  # even window length
    hop = max(n_win // 4, 1)
    Y_db = power_to_db(np.abs(stft(x, n_win, hop, window)) ** 2)

    fig = Figure(figsize=(9, 5))
    ax1, ax2 = fig.subplots(2, 1, sharex=True, gridspec_kw={"height_ratios": [1, 3]})
    ax1.plot(np.arange(len(x)) / sample_rate, np.clip(x, -2, 2), color="navy", linewidth=0.5)
    ax1.set_ylabel("Amplitude")
    plot_spectrogram(ax2, Y_db, sample_rate, n_win, hop, fmin=fmin, fmax=fmax)
    ax2.set_title(
        f"window: {n_win} samples = {1000 * n_win / sample_rate:.0f} ms | "
        f"frequency resolution: {sample_rate / n_win:.1f} Hz",
        fontsize=10,
    )
    fig.tight_layout()
    return fig


style = {"description_width": "100px"}
length_controls = {
    "signal_name": widgets.Dropdown(options=list(SIGNALS), description="signal", style=style, layout=widgets.Layout(width="450px")),
    "window_ms": widgets.SelectionSlider(options=[4, 8, 16, 32, 64, 128, 256, 512], value=32, description="length (ms)", style=style),
    "window": widgets.Dropdown(options=list(WINDOW_FUNCTIONS), value="hann", description="window", style=style),
}
ipd.display(widgets.VBox(list(length_controls.values())), interactive_figure(window_length_figure, length_controls))

With a short window (e.g., 8 ms), the two clicks appear as two separate vertical lines, but the two sinusoids are blurred into one thick horizontal band. With a long window (e.g., 256 ms), the two sinusoids appear as two separate horizontal lines, but the two clicks are blurred together. There is no window length that is best for everything: it is a trade-off, which is also known as the *uncertainty principle*.

For the piano example, short windows show when the notes start, but the harmonics of different notes are blurred. Long windows resolve the harmonics (and the different notes that are played at the same time), but the onsets are smeared in time. For music, windows of around 50–200 ms are a common compromise. As we will see next, to resolve *low notes* we need long windows.

## 4. Log-frequency spectrogram

The frequency axis of a spectrogram is linear: the bins are equally spaced in Hz. However, we perceive pitch on a *logarithmic* scale: each octave doubles the frequency. In Western music, pitches follow the equal-tempered scale, where an octave is divided into 12 semitones. If we number the pitches with MIDI pitch numbers $p$ (with A4 = 69 and C4, the middle C, = 60), the center frequency of pitch $p$ is

$$F_\mathrm{pitch}(p) = 2^{(p - 69)/12} \cdot 440~\mathrm{Hz}.$$

In the spectrogram above, the notes from the lower octaves are crowded at the bottom of the image, while the higher ones are spread over a large area. To get a representation in which the axis is labeled by pitches, we **pool** the spectrogram: we assign each frequency bin $k$ to the pitch $p$ with the closest center frequency, i.e., to the pitch for which the frequency $F(k)$ is between $F_\mathrm{pitch}(p - 0.5)$ and $F_\mathrm{pitch}(p + 0.5)$. Then, we sum the values of all the bins that belong to the same pitch:

$$\mathcal{Y}_\mathrm{LF}(m, p) = \sum_{k \in P(p)} \mathcal{Y}(m, k),$$

where $P(p)$ is the set of bins assigned to pitch $p$. The result is called a **log-frequency spectrogram**.

In [ ]:
def pitch_to_freq(p: float, pitch_ref: float = 69, freq_ref: float = 440.0) -> float:
    """Center frequency (in Hz) of a MIDI pitch in the equal-tempered scale"""
    return freq_ref * 2 ** ((p - pitch_ref) / 12)


def pitch_bins(p: int, sample_rate: float, window_length: int) -> np.ndarray:
    """Indices of the frequency bins of the STFT that are assigned to pitch p"""
    F = bin_frequencies(window_length, sample_rate)
    return np.where((F >= pitch_to_freq(p - 0.5)) & (F < pitch_to_freq(p + 0.5)))[0]


def log_freq_spectrogram(Y: np.ndarray, sample_rate: float, window_length: int) -> np.ndarray:
    """Log-frequency spectrogram, with shape (128, number of frames): MIDI pitches in the rows"""
    Y_LF = np.zeros((128, Y.shape[1]))
    for p in range(128):
        bins = pitch_bins(p, sample_rate, window_length)
        if len(bins) > 0:
            Y_LF[p] = Y[bins].sum(axis=0)
    return Y_LF


def note_name(p: int) -> str:
    """Name of a MIDI pitch (e.g., 60 -> C4)"""
    names = ["C", "C♯", "D", "D♯", "E", "F", "F♯", "G", "G♯", "A", "A♯", "B"]
    return f"{names[p % 12]}{p // 12 - 1}"


N, H = 4096, 512
print(f"Sample rate: {SAMPLE_RATE} Hz | window length: {N} samples | frequency resolution: {SAMPLE_RATE / N:.2f} Hz")
for p in [76, 64, 52, 40, 39, 38]:
    print(f"Pitch {p} ({note_name(p)}, {pitch_to_freq(p):6.1f} Hz): {len(pitch_bins(p, SAMPLE_RATE, N))} frequency bins")

For high pitches, many bins are assigned to each pitch. But for low pitches, the spacing between semitones (in Hz) is smaller than the frequency resolution of the STFT, so some pitches get only one bin or even *no bin at all*. Here, with a window of 4096 samples (186 ms), this already happens around E2 (pitch 40) and below. This is the price that we pay for the logarithmic axis: we need long windows to resolve low pitches. Let's compute the log-frequency spectrogram of the piano example and compare it with the spectrogram.

In [ ]:
X_piano = stft(x_piano, N, H)
Y_piano = np.abs(X_piano) ** 2
Y_LF = log_freq_spectrogram(Y_piano, SAMPLE_RATE, N)

# The pitches that we show, with a label for every C
p_min, p_max = 36, 96
frame_dur = H / SAMPLE_RATE
extent = [-frame_dur / 2, (Y_LF.shape[1] - 0.5) * frame_dur, p_min - 0.5, p_max + 0.5]
c_pitches = list(range(p_min, p_max + 1, 12))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_spectrogram(axes[0], power_to_db(Y_piano), SAMPLE_RATE, N, H, fmax=2000)
axes[0].set_title("Spectrogram (linear frequency axis)")
axes[1].imshow(power_to_db(Y_LF[p_min : p_max + 1]), origin="lower", aspect="auto", extent=extent, cmap="magma")
axes[1].set_yticks(c_pitches)
axes[1].set_yticklabels([note_name(p) for p in c_pitches])
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Pitch")
axes[1].set_title("Log-frequency spectrogram")
plt.tight_layout()
plt.show()

In the log-frequency spectrogram, the vertical axis is labeled by pitch, and the harmonics of a note are now at fixed distances above its fundamental (12 semitones for the 2nd harmonic, 19 for the 3rd, 24 for the 4th, ...), no matter what the note is. Note the black horizontal line near the bottom: it is a pitch (D♯2) that did not get any frequency bin, and the lowest pitches are noisy. This is the problem with the frequency resolution that we have just discussed.

## 5. Chromagram

Pitches that are one or several octaves apart sound very similar, and play the same role in harmony: we give them the same name (C, C♯, D, ...), the **pitch class** or *chroma*. To make our representation robust to differences in timbre, instrumentation and register (and to focus on the harmony), we sum the pitch bands that belong to the same pitch class (i.e., that have the same pitch modulo 12):

$$\mathcal{C}(m, c) = \sum_{p \,:\, p \bmod 12 = c} \mathcal{Y}_\mathrm{LF}(m, p), \qquad c \in [0:11],$$

with $c = 0$ for C, $c = 1$ for C♯, ..., $c = 11$ for B. The result is a **chromagram**: a sequence of 12-dimensional *chroma vectors*, which tells us how much energy there is in each of the 12 pitch classes at each time. Optionally, the chroma vectors can be **compressed** (with a logarithm, $\log(1 + \gamma\, \mathcal{C})$, where we first divide $\mathcal{C}$ by its maximum value; this makes the weaker notes more visible), and **normalized** (so that the loudness is not important, and only the distribution of the pitch classes remains).

In [ ]:
PITCH_CLASSES = ["C", "C♯", "D", "D♯", "E", "F", "F♯", "G", "G♯", "A", "A♯", "B"]


def chromagram(Y_LF: np.ndarray) -> np.ndarray:
    """Chromagram (shape (12, number of frames)) from a log-frequency spectrogram"""
    return np.stack([Y_LF[c::12].sum(axis=0) for c in range(12)])


def log_compress(C: np.ndarray, gamma: float = 100) -> np.ndarray:
    """Logarithmic compression log(1 + gamma * C), where C is first divided by its maximum"""
    return np.log(1 + gamma * C / max(np.max(C), 1e-12))


def normalize_columns(C: np.ndarray, threshold: float = 1e-6) -> np.ndarray:
    """Normalize each chroma vector to unit (Euclidean) norm. Silent frames become uniform vectors"""
    norms = np.linalg.norm(C, axis=0)
    uniform = np.ones(12) / np.sqrt(12)
    return np.where(norms > threshold, C / np.maximum(norms, threshold), uniform[:, None])


def plot_chromagram(ax: Axes, C: np.ndarray, sample_rate: float, hop_size: int, cmap: str = "magma") -> None:
    """Plot a chromagram: time on the x axis and pitch classes on the y axis"""
    frame_dur = hop_size / sample_rate
    extent = [-frame_dur / 2, (C.shape[1] - 0.5) * frame_dur, -0.5, 11.5]
    ax.imshow(C, origin="lower", aspect="auto", extent=extent, cmap=cmap)
    ax.set_yticks(range(12))
    ax.set_yticklabels(PITCH_CLASSES)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Pitch class")


C = chromagram(Y_LF)
C_compressed = log_compress(C, gamma=100)
C_normalized = normalize_columns(C_compressed)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
plot_chromagram(axes[0], C, SAMPLE_RATE, H)
axes[0].set_title("Chromagram")
plot_chromagram(axes[1], C_compressed, SAMPLE_RATE, H)
axes[1].set_title("Compressed ($\\gamma = 100$)")
plot_chromagram(axes[2], C_normalized, SAMPLE_RATE, H)
axes[2].set_title("Compressed and normalized")
plt.tight_layout()
plt.show()

Without compression, only the strongest notes are visible. After compression, we see the notes more clearly, and after normalization, the quieter parts get the same importance as the loud ones.

The following plot lets you explore the effect of the window length and the compression on the log-frequency spectrogram and the chromagram. What happens if the window is too short? And if the window is too long?

In [ ]:
def chroma_figure(window_length: int, gamma: float) -> Figure:
    """Figure with the log-frequency spectrogram and the chromagram of the piano example"""
    hop = 512
    Y = np.abs(stft(x_piano, window_length, hop)) ** 2
    Y_LF = log_freq_spectrogram(Y, SAMPLE_RATE, window_length)
    C = chromagram(Y_LF)
    C = normalize_columns(log_compress(C, gamma) if gamma > 0 else C)

    frame_dur = hop / SAMPLE_RATE
    p_min, p_max = 36, 96
    extent = [-frame_dur / 2, (Y.shape[1] - 0.5) * frame_dur, p_min - 0.5, p_max + 0.5]
    fig = Figure(figsize=(9, 6))
    ax1, ax2 = fig.subplots(2, 1, sharex=True)
    Y_LF_show = log_compress(Y_LF[p_min : p_max + 1], gamma) if gamma > 0 else Y_LF[p_min : p_max + 1]
    ax1.imshow(Y_LF_show, origin="lower", aspect="auto", extent=extent, cmap="magma")
    ax1.set_yticks(range(p_min, p_max + 1, 12))
    ax1.set_yticklabels([note_name(p) for p in range(p_min, p_max + 1, 12)])
    ax1.set_ylabel("Pitch")
    ax1.set_title(f"Log-frequency spectrogram (window: {window_length} samples = {1000 * window_length / SAMPLE_RATE:.0f} ms)", fontsize=10)
    plot_chromagram(ax2, C, SAMPLE_RATE, hop)
    ax2.set_title("Chromagram (normalized)", fontsize=10)
    fig.tight_layout()
    return fig


chroma_controls = {
    "window_length": widgets.SelectionSlider(options=[512, 1024, 2048, 4096, 8192], value=4096, description="window length", style={"description_width": "100px"}),
    "gamma": widgets.SelectionSlider(options=[0, 1, 10, 100, 1000, 10000], value=100, description="compression $\\gamma$", style={"description_width": "100px"}),
}
ipd.display(widgets.VBox(list(chroma_controls.values())), interactive_figure(chroma_figure, chroma_controls))

With a short window, the low notes are not resolved, and their energy goes to the wrong pitch classes. With a very long window, the chromagram becomes blurred in time. A window of about 4096 samples is a good compromise for this recording. Without compression ($\gamma = 0$), the chromagram is dominated by a few strong notes.

### Comparing the chromagram with the score

The chromagram is the audio counterpart of something that we know from symbolic music: a **pitch-class piano roll**, which we can compute from the score. Below, we show the chromagram of the performance next to the pitch-class piano roll of its score (computed with partitura). Both show the same music, and they have the same structure, but they are in different "time units": seconds for the audio, and beats for the score. This is why we need *alignment*, which is the topic of the next notebooks!

In [ ]:
score = pt.load_score(os.path.join(MEC_VARIATION_DIR, "scores", "musicxml", "snippet.musicxml"))
score_pc = pt.utils.music.compute_pitch_class_pianoroll(score, time_unit="beat", time_div=8)
n_beats = score_pc.shape[1] / 8

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
plot_chromagram(axes[0], C_normalized, SAMPLE_RATE, H)
axes[0].set_title("Chromagram of the performance")
axes[1].imshow(score_pc, origin="lower", aspect="auto", extent=[0, n_beats, -0.5, 11.5], cmap="magma")
axes[1].set_xlabel("Time (beats)")
axes[1].set_title("Pitch-class piano roll of the score")
plt.tight_layout()
plt.show()

### Comparison with librosa

Many libraries compute chromagrams. For example, `librosa.feature.chroma_stft` computes a chromagram from a spectrogram. It uses a slightly different procedure (it distributes the energy of each frequency bin over the pitch classes with a smooth weighting, instead of assigning each bin to one pitch), so the result is not identical to ours, but it shows the same structure:

In [ ]:
C_librosa = librosa.feature.chroma_stft(S=Y_piano, sr=SAMPLE_RATE, n_fft=4096, tuning=0.0, norm=2)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
plot_chromagram(axes[0], C_normalized, SAMPLE_RATE, H)
axes[0].set_title("Our chromagram")
plot_chromagram(axes[1], C_librosa, SAMPLE_RATE, H)
axes[1].set_title("librosa.feature.chroma_stft")
plt.tight_layout()
plt.show()

## Summary

- The Fourier transform of a whole signal does not tell us *when* frequencies occur. The **STFT** solves this by computing Fourier transforms of short sections of the signal, which we cut out with a **window**. Its squared magnitude is the **spectrogram**.
- The **window length** is a trade-off between time and frequency resolution, and the **window shape** controls the leakage in the spectrum.
- A **log-frequency spectrogram** assigns the frequency bins to the pitches of the equal-tempered scale. Low pitches need long windows.
- A **chromagram** folds the pitches into the 12 pitch classes. It is robust to timbre and register, and it is a very useful feature for music alignment, because it can be compared with symbolic representations (like the pitch-class piano roll) and with other recordings.

**Try it yourself:**

- Record your own sound with the last example of the Fourier analysis notebook, or use the full recording `short1.wav` of the `mec-variation` dataset, and compute its spectrogram and chromagram.
- Change the hop size of the STFT. What happens to the time resolution and to the size of the spectrogram?
- What does the chromagram of a single note look like? What about a chord?

## Acknowledgment and references

This notebook is based on the FMP notebooks linked at the beginning, which were created by Meinard Müller and Frank Zalkow. The music example comes from the `mec-variation` dataset (see the README of this folder for more information and its license).

[1] Meinard Müller: *Fundamentals of Music Processing – Using Python and Jupyter Notebooks*. Springer Verlag, 2nd edition, 2021. <https://www.music-processing.de/>

[2] Meinard Müller and Frank Zalkow: *FMP Notebooks: Educational Material for Teaching and Learning Fundamentals of Music Processing*. Proceedings of the International Conference on Music Information Retrieval (ISMIR), pp. 573–580, Delft, The Netherlands, 2019. <https://www.audiolabs-erlangen.de/FMP>

[3] Meinard Müller and Frank Zalkow: *libfmp: A Python Package for Fundamentals of Music Processing*. Journal of Open Source Software (JOSS), 6(63), 2021.